In [1]:
import torch
import pandas as pd
from pathlib import Path
import numpy as np

# Carregar DataFrames com todas as tabelas
edstays_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/edstays.csv')
edstays_df = pd.read_csv(edstays_path)

diagnosis_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/diagnosis.csv')
diagnosis_df = pd.read_csv(diagnosis_path)

medrecon_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/medrecon.csv')
medrecon_df = pd.read_csv(medrecon_path)

pyxis_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/pyxis.csv')
pyxis_df = pd.read_csv(pyxis_path)

triage_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/triage.csv')
triage_df = pd.read_csv(triage_path)

vitalsign_path = Path('physionet.org/files/mimic-iv-ed/2.2/ed/vitalsign.csv')
vitalsign_df = pd.read_csv(vitalsign_path)

patients_path = Path('physionet.org/files/mimiciv/3.0/hosp/patients.csv')
patients_df = pd.read_csv(patients_path)

In [2]:
data_df = pd.DataFrame()
data_df["id"] = edstays_df["stay_id"]
data_df["subject_id"] = edstays_df["subject_id"]

In [3]:
# Codificar dados categóricos
## Selecionar variáveis categóricas
data_df["gender"] = edstays_df["gender"]
data_df["race"] = edstays_df["race"]
data_df["arrival_transport"] = edstays_df["arrival_transport"]
categoric_cols = ["gender", "race", "arrival_transport"]

In [4]:
from sklearn import preprocessing 

label_encoders = {}

for var in categoric_cols:
    label_encoder = preprocessing.LabelEncoder()
    data_df[var] = label_encoder.fit_transform(data_df[var])
    label_encoders[var] = label_encoder

label_encoders

{'gender': LabelEncoder(),
 'race': LabelEncoder(),
 'arrival_transport': LabelEncoder()}

In [5]:
# Incluir informações sobre a hora, o dia da semana e o mês do ano em que a visita ao serviço de emergência ocorreu
data_df['intime'] = pd.to_datetime(edstays_df['intime'])
data_df['in_day_of_the_week'] = data_df['intime'].dt.day_of_week
data_df['in_hour_of_the_day'] = data_df['intime'].dt.hour
data_df['in_month_of_the_year'] = data_df['intime'].dt.month

In [6]:
# Circular encoding - Padronizar dados cíclicos para representar melhor a ciclicidade presente no dado
## Selecionar variáveis cíclicas
cyclic_cols = ['in_day_of_the_week', 'in_hour_of_the_day', 'in_month_of_the_year']
## Realizar circular encoding para cada variável
for col in cyclic_cols:
  data_df[f'{col}_sin'] = np.sin(2 * np.pi * data_df[col] / len(data_df[col].unique()))
  data_df[f'{col}_cos'] = np.cos(2 * np.pi * data_df[col] / len(data_df[col].unique()))
## Deletar colunas originais
data_df.drop(columns=cyclic_cols, inplace=True)

In [7]:
# Incluir se o paciente foi admitido ao hospital ou não
data_df['admitted_to_hosp'] = edstays_df['hadm_id'].apply(lambda x: np.isnan(x)==False).astype(int)


In [8]:
# Incluir dados coletados na triagem do hospital
## Seleção das variáveis relevantes
triage_data_df = triage_df[['stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity', 'pain']]

## Conversão dos dados da coluna pain para dado numérico inteiro quando possível. Se não é possível, deixar como missing.
def convert_to_int(value):
    try:
        return int(value)
    except:
        return np.nan
    
triage_data_df['pain'] = triage_data_df['pain'].apply(convert_to_int)

## União das tabelas
data_df = pd.merge(data_df, triage_data_df, left_on='id', right_on='stay_id', how='left')

C:\Users\Vinicius\AppData\Local\Temp\ipykernel_34152\2544293574.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  triage_data_df['pain'] = triage_data_df['pain'].apply(convert_to_int)


In [9]:
# Incluir dado de idade do paciente ('anchor_age')
patients_age_df = patients_df[['subject_id', 'anchor_age']]
## União das tabelas através do id do paciente
data_df = pd.merge(data_df, patients_age_df, left_on='subject_id', right_on='subject_id', how='left')

In [10]:
# Pegar lista de códigos etccode de medicamentos e associar a cada stay_id
med_by_stay_id_etccode = medrecon_df.groupby('stay_id')['etccode'].unique()
data_df = pd.merge(data_df, med_by_stay_id_etccode, left_on='id', right_on='stay_id', how='left')

# Pegar lista de códigos NDC (National Drug Code) de medicamentos e associar a cada stay_id
med_by_stay_id_ndc = medrecon_df.groupby('stay_id')['ndc'].unique()
data_df = pd.merge(data_df, med_by_stay_id_ndc, left_on='id', right_on='stay_id', how='left')

In [11]:
# Incluir número de medicamentos que o paciente usava antes da passagem na emergência

def get_len(list):
    if list is np.nan:
        return 0
    else:
        return len(list)

# Pegar o número de medicamentos segundo códigos NDC únicos 
data_df['med_count_by_ndc'] = data_df['ndc'].apply(get_len)

In [12]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425087 entries, 0 to 425086
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   id                        425087 non-null  int64         
 1   subject_id                425087 non-null  int64         
 2   gender                    425087 non-null  int32         
 3   race                      425087 non-null  int32         
 4   arrival_transport         425087 non-null  int32         
 5   intime                    425087 non-null  datetime64[ns]
 6   in_day_of_the_week_sin    425087 non-null  float64       
 7   in_day_of_the_week_cos    425087 non-null  float64       
 8   in_hour_of_the_day_sin    425087 non-null  float64       
 9   in_hour_of_the_day_cos    425087 non-null  float64       
 10  in_month_of_the_year_sin  425087 non-null  float64       
 11  in_month_of_the_year_cos  425087 non-null  float64       
 12  ad

In [13]:
columns_to_drop = ['id', 'subject_id', 'intime', 'stay_id', 'etccode', 'ndc']

numeric_cols = ['anchor_age','temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity', 'pain', 'med_count_by_ndc']

In [14]:
from sklearn.model_selection import train_test_split

# Preparar datasets
random_state = 42

## Eliminar dados faltantes agora
data_df = data_df.dropna()

## Separar variáveis preditoras da variável a ser predita
X = data_df.drop('admitted_to_hosp', axis=1)
y = data_df['admitted_to_hosp']
## Separar em dataset de treino (90%) e teste (10%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=random_state)
## Separar dados de treino em treino (90%) e validação (10%)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=random_state)

In [15]:
# Ajustar valores claramente equivocados nas variáveis
## temperature
X_train.loc[X_train['temperature'] > 200, 'temperature'] = np.nan
X_val.loc[X_val['temperature'] > 200, 'temperature'] = np.nan
X_test.loc[X_test['temperature'] > 200, 'temperature'] = np.nan

In [16]:
## heartrate
X_train.loc[X_train['heartrate'] > 400, 'heartrate'] = np.nan
X_val.loc[X_val['heartrate'] > 400, 'heartrate'] = np.nan
X_test.loc[X_test['heartrate'] > 400, 'heartrate'] = np.nan

In [17]:
## resprate
X_train.loc[X_train['resprate'] > 60, 'resprate'] = np.nan
X_val.loc[X_val['resprate'] > 60, 'resprate'] = np.nan
X_test.loc[X_test['resprate'] > 60, 'resprate'] = np.nan

In [18]:
## o2sat
X_train.loc[X_train['o2sat'] > 100, 'o2sat'] = np.nan
X_val.loc[X_val['o2sat'] > 100, 'o2sat'] = np.nan
X_test.loc[X_test['o2sat'] > 100, 'o2sat'] = np.nan

In [19]:
## sbp
X_train.loc[X_train['sbp'] > 300, 'sbp'] = np.nan
X_val.loc[X_val['sbp'] > 300, 'sbp'] = np.nan
X_test.loc[X_test['sbp'] > 300, 'sbp'] = np.nan

In [20]:
## dbp
X_train.loc[X_train['dbp'] > 300, 'dbp'] = np.nan
X_val.loc[X_val['dbp'] > 300, 'dbp'] = np.nan
X_test.loc[X_test['dbp'] > 300, 'dbp'] = np.nan

In [21]:
## pain - Considerar apenas valores entre 0 e 10
X_train.loc[(X_train['pain'] > 10)|(X_train['pain'] < 0), 'pain'] = np.nan
X_val.loc[(X_val['pain'] > 10)|(X_val['pain'] < 0), 'pain'] = np.nan
X_test.loc[(X_test['pain'] > 10)|(X_test['pain'] < 0), 'pain'] = np.nan

In [22]:
# Normalizar dados numéricos
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
## Dados de treino
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
## Aplicar o mesmo scaler aos dados de validação e teste
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [23]:
# tabelas de frequências com porcentagens, se quiser com frequencias absolutas tirar o normalize
for col in categoric_cols:
    print(X_train[col].value_counts(normalize=True).sort_index(), end='\n\n')

gender
0    0.574697
1    0.425303
Name: proportion, dtype: float64

race
0     0.002275
1     0.012027
2     0.003333
3     0.014642
4     0.001217
5     0.003412
6     0.009245
7     0.179118
8     0.016952
9     0.008456
10    0.006723
11    0.001662
12    0.002914
13    0.002019
14    0.018557
15    0.004823
16    0.001927
17    0.002368
18    0.033975
19    0.002972
20    0.000507
21    0.001093
22    0.040407
23    0.001234
24    0.003655
25    0.002385
26    0.000154
27    0.004043
28    0.570843
29    0.003148
30    0.003430
31    0.024337
32    0.016145
Name: proportion, dtype: float64

arrival_transport
0    0.361869
1    0.000194
2    0.002509
3    0.010405
4    0.625024
Name: proportion, dtype: float64



In [24]:
from sklearn.impute import SimpleImputer

## Criar coluna com one-hot encoding definindo se existe valor faltando ou não para aquela variável
cols_with_missing_values = []
for col in data_df.columns:
  if data_df[col].isna().any():
    cols_with_missing_values.append(col)

print(cols_with_missing_values)

# Realizar substuição de dados faltantes com a média dos dados no dataset de treino
imputer = SimpleImputer(strategy='mean')
## Preparar imputer com dados de treino
cols_with_missing_values = [col for col in set([*cols_with_missing_values, *numeric_cols]) if col in X_train.columns]
imputer.fit(X_train[cols_with_missing_values])
## Aplicar em todos os datasets (treino, validação e teste)
X_train[cols_with_missing_values] = imputer.transform(X_train[cols_with_missing_values])
X_val[cols_with_missing_values] = imputer.transform(X_val[cols_with_missing_values])
X_test[cols_with_missing_values] = imputer.transform(X_test[cols_with_missing_values])

[]


In [25]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 280022 entries, 0 to 425084
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   id                        280022 non-null  int64         
 1   subject_id                280022 non-null  int64         
 2   gender                    280022 non-null  int32         
 3   race                      280022 non-null  int32         
 4   arrival_transport         280022 non-null  int32         
 5   intime                    280022 non-null  datetime64[ns]
 6   in_day_of_the_week_sin    280022 non-null  float64       
 7   in_day_of_the_week_cos    280022 non-null  float64       
 8   in_hour_of_the_day_sin    280022 non-null  float64       
 9   in_hour_of_the_day_cos    280022 non-null  float64       
 10  in_month_of_the_year_sin  280022 non-null  float64       
 11  in_month_of_the_year_cos  280022 non-null  float64       
 12  admitte

In [26]:
datasets = [X_train, X_val, X_test]
for dataset in datasets:
  dataset.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Salvar no disco os datasets
X_train.to_csv(Path('gnn_data', 'X_train.csv'))
X_val.to_csv(Path('gnn_data', 'X_val.csv'))
X_test.to_csv(Path('gnn_data', 'X_test.csv'))

y_train.to_csv(Path('gnn_data', 'y_train.csv'))
y_val.to_csv(Path('gnn_data', 'y_val.csv'))
y_test.to_csv(Path('gnn_data', 'y_test.csv'))

print("Train set size:", X_train.shape)
print("Validation set size:", X_val.shape)
print("Test set size:", X_test.shape)


Train set size: (226817, 19)
Validation set size: (25202, 19)
Test set size: (28003, 19)


In [27]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226817 entries, 388904 to 269659
Data columns (total 19 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   gender                    226817 non-null  int32  
 1   race                      226817 non-null  int32  
 2   arrival_transport         226817 non-null  int32  
 3   in_day_of_the_week_sin    226817 non-null  float64
 4   in_day_of_the_week_cos    226817 non-null  float64
 5   in_hour_of_the_day_sin    226817 non-null  float64
 6   in_hour_of_the_day_cos    226817 non-null  float64
 7   in_month_of_the_year_sin  226817 non-null  float64
 8   in_month_of_the_year_cos  226817 non-null  float64
 9   temperature               226817 non-null  float64
 10  heartrate                 226817 non-null  float64
 11  resprate                  226817 non-null  float64
 12  o2sat                     226817 non-null  float64
 13  sbp                       226817 non-null  f

In [28]:
time_related_columns = ['in_day_of_the_week_sin',
                        'in_day_of_the_week_cos',
                        'in_hour_of_the_day_sin',
                        'in_hour_of_the_day_cos',
                        'in_month_of_the_year_sin',
                        'in_month_of_the_year_cos']

vital_signs_columns = [
    'temperature',
    'heartrate',
    'resprate',
    'o2sat',
    'sbp',
    'dbp',
]

In [40]:
X_test[vital_signs_columns].info()

<class 'pandas.core.frame.DataFrame'>
Index: 28003 entries, 239532 to 209502
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   temperature  28003 non-null  float64
 1   heartrate    28003 non-null  float64
 2   resprate     28003 non-null  float64
 3   o2sat        28003 non-null  float64
 4   sbp          28003 non-null  float64
 5   dbp          28003 non-null  float64
dtypes: float64(6)
memory usage: 1.5 MB


In [35]:
torch.tensor(X_train.loc[vital_signs_columns])

KeyError: "None of [Index(['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp'], dtype='object')] are in the [index]"

In [3]:
import torch
from torch_geometric.data import InMemoryDataset
from tqdm import tqdm 
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
import torch_geometric.transforms as T
import pandas as pd
from pathlib import Path


class MimicIVDataset(InMemoryDataset):
    def __init__(self, root: Path, x_filename: Path, y_filename: Path, transform=None, pre_transform=None, pre_filter=None):
        
        self.root = root
        self.x_filename = x_filename
        self.y_filename = y_filename
        
        super().__init__(root, transform, pre_transform, pre_filter)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return [self.x_filename, self.y_filename]

    @property
    def processed_file_names(self):
        return [f'{self.x_filename}.pt']
    
    def process(self):
        # Read files
        X = pd.read_csv(self.root / self.x_filename, index_col=0)
        y = pd.read_csv(self.root / self.y_filename, index_col=0)

        # Read data into huge `Data` list.
        data_list = []
        for idx in tqdm(range(len(X))):
            sample = X.iloc[idx]
            sample_y = y_train.iloc[idx]
            
            data = HeteroData()

            # Defining nodes
            data['gender'].x = torch.tensor(sample['gender'], dtype=torch.long).view(1, -1)
            data['race'].x = torch.tensor(sample['race'], dtype=torch.long).view(1, -1)
            data['arrival_transport'].x = torch.tensor(sample['arrival_transport'], dtype=torch.long).view(1, -1)
            data['anchor_age'].x = torch.tensor(sample['anchor_age']).view(1, -1)
            data['vital_signs'].x = torch.tensor(sample[vital_signs_columns].values).view(1, -1)
            data['person'].x = torch.tensor(0, dtype=torch.long).view(1, -1)

            # Defining edges
            data['gender','person'].edge_index = torch.tensor([[0, 0]]).t().contiguous()
            data['race','person'].edge_index = torch.tensor([[0, 0]]).t().contiguous()
            data['arrival_transport','person'].edge_index = torch.tensor([[0, 0]]).t().contiguous()
            data['anchor_age','person'].edge_index = torch.tensor([[0, 0]]).t().contiguous()
            data['vital_signs','person'].edge_index = torch.tensor([[0, 0]]).t().contiguous()

            data.y = torch.tensor(sample_y).view(-1)

            data_list.append(data)

        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        self.save(data_list, self.processed_paths[0])


# Define transforms
transform = T.Compose([T.ToUndirected(), T.AddSelfLoops()])

# Load datasets
train_dataset = MimicIVDataset(root=Path('gnn_data'), x_filename=Path('X_train.csv'), y_filename=Path('y_train.csv'), transform=transform)
val_dataset = MimicIVDataset(root=Path('gnn_data'), x_filename=Path('X_val.csv'), y_filename=Path('y_val.csv'), transform=transform)
test_dataset = MimicIVDataset(root=Path('gnn_data'), x_filename=Path('X_test.csv'), y_filename=Path('y_test.csv'), transform=transform)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
train_dataset[0]

HeteroData(
  y=[1],
  gender={ x=[1, 1] },
  race={ x=[1, 1] },
  arrival_transport={ x=[1, 1] },
  anchor_age={ x=[1, 1] },
  person={ x=[1, 1] },
  (gender, to, person)={ edge_index=[2, 1] },
  (race, to, person)={ edge_index=[2, 1] },
  (arrival_transport, to, person)={ edge_index=[2, 1] },
  (anchor_age, to, person)={ edge_index=[2, 1] },
  (person, rev_to, gender)={ edge_index=[2, 1] },
  (person, rev_to, race)={ edge_index=[2, 1] },
  (person, rev_to, arrival_transport)={ edge_index=[2, 1] },
  (person, rev_to, anchor_age)={ edge_index=[2, 1] }
)

In [11]:
import torch
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv  # You can also use GCNConv, GATConv, etc.
from torch_geometric.data import HeteroData

class HeteroGraphModel(torch.nn.Module):
    def __init__(self, num_categories, embedding_dim, hidden_layer_dim = 32):
        super(HeteroGraphModel, self).__init__()
        
        # Embedding layers for each node type
        self.gender_emb = nn.Embedding(num_categories['gender'], embedding_dim)
        self.race_emb = nn.Embedding(num_categories['race'], embedding_dim)
        self.arrival_transport_emb = nn.Embedding(num_categories['arrival_transport'], embedding_dim)
        self.person_emb = nn.Embedding(num_categories['person'], embedding_dim)

        # HeteroConv for message passing between node types
        self.conv1 = HeteroConv({
            ('gender', 'to', 'person'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('race', 'to', 'person'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('arrival_transport', 'to', 'person'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('anchor_age', 'to', 'person'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('person', 'rev_to', 'gender'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('person', 'rev_to', 'race'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('person', 'rev_to', 'arrival_transport'): SAGEConv(embedding_dim, hidden_layer_dim),
            ('person', 'rev_to', 'anchor_age'): SAGEConv(embedding_dim, hidden_layer_dim),
        })
        
        self.conv2 = HeteroConv({
            ('gender', 'to', 'person'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('race', 'to', 'person'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('arrival_transport', 'to', 'person'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('anchor_age', 'to', 'person'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('person', 'rev_to', 'gender'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('person', 'rev_to', 'race'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('person', 'rev_to', 'arrival_transport'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
            ('person', 'rev_to', 'anchor_age'): SAGEConv(hidden_layer_dim, hidden_layer_dim),
        })

        # Linear projections
        self.linear_proj_anchor_age = nn.Sequential(
            nn.Linear(1, embedding_dim),
            nn.ReLU(),
            )
        
        self.linear_proj_vital_signs = nn.Sequential(
            nn.Linear(6, embedding_dim),
            nn.ReLU(),
            )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_layer_dim, 1),
            nn.Sigmoid(),
            )

    def forward(self, data: HeteroData):

        # Apply embeddings to each node type
        gender_emb = self.gender_emb(data['gender'].x).squeeze(1)
        race_emb = self.race_emb(data['race'].x).squeeze(1)
        arrival_transport_emb = self.arrival_transport_emb(data['arrival_transport'].x).squeeze(1)
        person_emb = self.person_emb(torch.zeros(data['person'].x.size(), dtype=torch.long).to(device)).squeeze(1)
        age_emb = self.linear_proj_anchor_age(data['anchor_age'].x)

        # Combine the embeddings in the HeteroData object
        x_dict = {
            'gender': gender_emb,
            'race': race_emb,
            'arrival_transport': arrival_transport_emb,
            'person': person_emb,
            'anchor_age': age_emb,
        }

        # Perform message passing using HeteroConv
        out_dict = self.conv1(x_dict, data.edge_index_dict)
        out_dict = self.conv2(x_dict, data.edge_index_dict)
        prediction = self.classifier(out_dict["person"])

        return out_dict, prediction
    



In [12]:
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC

# Prepare dataloaders
batch_size = 1000

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=True),
    'test': DataLoader(test_dataset, batch_size=batch_size, shuffle=True),
}

dataset_sizes = {'train': len(train_dataset),
                 'val': len(val_dataset),
                 'test': len(test_dataset)}

print(f'dataset_sizes = {dataset_sizes}')

# Instantiation
num_categories = {
    'gender': 2,  
    'race': 33,    
    'arrival_transport': 5, 
    'person': 1 
}

embedding_dim = 10
num_epochs = 1
learning_rate = 0.01

model = HeteroGraphModel(num_categories, embedding_dim).to(torch.float64)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.BCELoss()

threshold = 0.5
acc = BinaryAccuracy(threshold=threshold).to(device)
auc = BinaryAUROC().to(device)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch_idx, data in enumerate(dataloader):
        data = data.to(device)

        # Compute prediction error
        out, pred = model(data)
        loss = loss_fn(pred.view(-1), data.y.view(-1).to(torch.float64))

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_idx % 20 == 0:
            loss = loss.detach()
            current = batch_idx * batch_size
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(model, loader):
     model.eval()

     # Reset estimators
     acc.reset()
     auc.reset()

     for data in loader:  # Iterate in batches over the training/test dataset.
         out, pred = model(data.to(device))  
         acc(pred.view(-1), data.y.view(-1).to(torch.float64))
         auc(pred.view(-1), data.y.view(-1).to(torch.float64))

     return acc.compute(), auc.compute()  # Derive ratio of correct predictions.


for epoch in range(num_epochs):
    print(f'\nEpoch: {epoch+1:03d} -----------------------------------------------------')
    train(dataloaders['train'], model, criterion, optimizer)
    if epoch % 1 == 0:
        val_acc, val_auc = test(model, dataloaders['val'])
        test_acc, test_auc = test(model, dataloaders['test'])
        print(f'Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')
        print(f'Val Auc: {val_auc:.4f}, Test Auc: {test_auc:.4f}')

dataset_sizes = {'train': 344320, 'val': 38258, 'test': 42509}

Epoch: 001 -----------------------------------------------------
loss: 0.773050  [    0/344320]
loss: 0.698995  [20000/344320]
loss: 0.692513  [40000/344320]
loss: 0.691344  [60000/344320]
loss: 0.692450  [80000/344320]
loss: 0.692531  [100000/344320]
loss: 0.693827  [120000/344320]
loss: 0.693109  [140000/344320]
loss: 0.692626  [160000/344320]
loss: 0.692866  [180000/344320]
loss: 0.692642  [200000/344320]
loss: 0.693242  [220000/344320]
loss: 0.693162  [240000/344320]
loss: 0.689884  [260000/344320]
loss: 0.694212  [280000/344320]
loss: 0.693591  [300000/344320]
loss: 0.687832  [320000/344320]
loss: 0.691067  [340000/344320]
Val Acc: 0.5286, Test Acc: 0.5277
Val Auc: 0.5000, Test Auc: 0.5000


In [13]:
test(model, dataloaders['train'])

(tensor(0.5224, device='cuda:0'), tensor(0.5000, device='cuda:0'))